# ALOE Engine Verification & Empirical Benchmark on `LiquidAI/LFM3-1.7B` (Google Colab GPU)

**Project:** *Adaptive LoRA Rank & Module-Group Allocation Across Heterogeneous Small Language Model Architectures* (`Walapalam/aloe`)  
**Model C (3rd Architecture Family):** `LiquidAI/LFM3-1.7B` (`Lfm3ForCausalLM`, 24-layer Sequential Tri-Hybrid: 13 Gated Short-Convolution `conv` layers + 5 Sliding-Window Local Attention layers + 6 Full-Context Global Attention layers + 24 SwiGLU FFN blocks)

### What This Notebook Verifies on Google Colab GPU:
1. **Full Automated Test Suite (`pytest`):** Runs all 107+ unit and adversarial tests across `Gemma-4-E2B`, `Gemma-4-E4B`, `Falcon-H1-1.5B`, and `LiquidAI/LFM3-1.7B`.
2. **Zero-RAM `meta`-Device Architecture Inspection (`aloe inspect`):** Verifies that ALOE derives a **4-group** search grammar (`local_attention`: 20, `global_attention`: 24, `ssm`: 26, `ffn`: 72 $\rightarrow$ **142 valid linear targets**, **145 excluded modules**, search space cardinality **194,480**) without modifying a single line of ALOE's search-space, executor, or BO core (SQ2 / RO4 extensibility proof).
3. **Pre-Training $\pm 5\%$ Parameter-Budget Parity (`FR04`):** Verifies exact analytical parameter counts against PEFT runtime counts across `P1 (Uniform r=8: 8,052,736 params)`, `P2_Global_Exact_Parity (r_loc=8, r_glb=16, r_ssm=4, r_ffn=8: 8,052,736 params, 0.00% delta)`, and `P3_Operator_Prioritized (r_loc=16, r_glb=16, r_ssm=16, r_ffn=4: 7,700,480 params, -4.37% delta)`.
4. **Pre-Flight Gradient Trap (`verify_gradients`) & Mandatory Negative Control:** Verifies that all 142 valid targets (284 LoRA tensors) achieve `GRADIENT_OK` (`norm > 1e-8`), while attaching LoRA to LFM3's 13 depthwise `conv.conv` kernels under packed SFT triggers `26/26 GRADIENT_MISSING` (`CONFIRMED_INERT`).
5. **Live GPU Budget-Matched SFT & AST Tool-Calling Evaluation:** Trains and evaluates `P1`, `P2`, and `P3` on unseen xLAM validation tools.

In [ ]:
# 1. Clone Walapalam/aloe and install dependencies on Google Colab
!rm -rf /content/aloe
!git clone https://github.com/Walapalam/aloe.git /content/aloe
%cd /content/aloe
!pip install -q -U "transformers>=5.15.0" peft accelerate datasets botorch gpytorch pytest psutil huggingface_hub pydantic typer rich
!pip install -q -e .

In [ ]:
# 2. Verify GPU hardware and run ALOE core/differentiation/budget/architecture unit tests
import torch
print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

!pytest tests/test_core_isolation.py tests/test_differentiation.py tests/test_budget_matching.py tests/test_grammar.py tests/test_trial_roundtrip.py tests/test_invalid_config_aborts.py tests/test_tool_calling_evaluator.py tests/test_bootstrap.py tests/test_eval_card.py tests/test_optimization.py tests/test_bo_adapter.py tests/test_xlam_split.py -v

In [ ]:
# 3. Zero-RAM Meta-Device Inspection of LiquidAI/LFM3-1.7B via CLI
!python -m aloe.cli.main inspect LiquidAI/LFM3-1.7B --out runs/grammars/lfm3_1_7b.json

In [ ]:
# 4. Full-Weight LiquidAI/LFM3-1.7B Pre-Flight Gradient Verification, Negative Control & Budget-Matched SFT+AST Pilot
!python scripts/run_colab_lfm3_tests.py --full-weights --run-sft-pilot --steps 25 --out runs/gradients/LiquidAI--LFM3-1.7B/colab_gpu_report.json

In [ ]:
# 5. Display Complete JSON Verification & SFT Pilot Report
import json
from pathlib import Path
report = json.loads(Path('runs/gradients/LiquidAI--LFM3-1.7B/colab_gpu_report.json').read_text())
print(json.dumps({
    'model_id': report['model_id'],
    'device': report['device'],
    'hardware': report['hardware'],
    'architecture': report['architecture'],
    'budget_parity': report['budget_parity'],
    'negative_control': report['gradient_verification']['negative_control_conv_conv'],
    'sft_pilot': report['sft_pilot'],
}, indent=2))